In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm

from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor


In [2]:
df_final = pd.read_excel("data/processed/df_final.xlsx")
df_final = df_final.sort_values("Date").reset_index(drop=True) 

In [3]:
covariates = ["dolvol_lag2", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "rd_mve", "sp", "nincr"]

I. Fonctions : normalisation et gestion des NaN

In [4]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=306):
    """
    Génère des splits temporels selon la logique décrite :
    - Train cumulatif (augmente d'un an à chaque refit)
    - Validation = fenêtre fixe glissante de 1 an 
    - Test = fenêtre fixe après la validation
    - Avance de step_months à chaque itération : 12 mois

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop si on n'a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Avancer d'un step (ex : 12 mois) pour le prochain refit
        start += step_months

    return splits

In [5]:
def preprocess_split(X_train, X_val, X_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).

    Paramètres
    ----------
    X_train, X_val, X_test : DataFrames bruts (avec 'Ticker' et 'Date')
    covariates : liste des colonnes numériques à traiter

    Retour
    ------
    X_train_scaled, X_val_scaled, X_test_scaled : DataFrames transformés (covariates seulement)
    """
    #Gestion des NaN : moyenne par Ticker calculée sur le train
    means_by_ticker = x_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for col in covariates:
            #On remplace NaN par la moyenne du ticker
            df[col] = df.apply(
                lambda row: means_by_ticker[col][row["Ticker"]] 
                            if pd.isna(row[col]) and row["Ticker"] in means_by_ticker.index 
                            else row[col],
                axis=1
            )
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [6]:
#Mesures : 
#R²
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    """
    Calcule le success ratio = proportion de signes correctement prédits.
    y_true et y_pred doivent être des array-like de même longueur.
    ignore_zero : si True, on ignore les observations où y_true == 0.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  # au cas où toutes les valeurs sont nulles

    return (sign_true == sign_pred).mean()


In [7]:
#Permet de récupérer x et y 
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target]) #garde toutes les colonnes mais enlève excess return
    y = subset[target]
    return x, y

In [8]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

100%|██████████| 3/3 [00:41<00:00, 13.79s/it]


BENCHMARK

In [9]:
"""
OLS : Ordinary Least Squares : Nous détaillons ici cet algorithme, la logique étant identique pour les autres modèles.
Listes utilisées : 
- r2_in_sample_list et r2_test_list : stockent, pour chaque split, les R² in‑sample et out‑of‑sample. Elles servent à analyser
  la performance split par split et à ajuster le tuning des modèles (utile pour les modèles à hyperparamètres).
- y_true : valeurs réelles de l’equity premium sur l’ensemble.
- y_pred_ols : prédictions correspondantes du modèle OLS. 
- y_trainval : données d’entraînement (train + validation) utilisées pour l’ajustement du modèle.
- dates_ols et tickers_ols : récupérées à chaque split pour pouvoir fusionner correctement les prédictions de tous les modèles
  et s’assurer que les lignes (dates/tickers) correspondent, évitant tout mélange potentiel des prédictions.

Df et résultats en sortie : 
- df_results_ols : df contenant les prédictions du modèles ols ainsi que la date et le ticker correspondant. 
- r2_results : dictionnaire contenant le r2 ols in sample et oos
"""

#pour calculer les r² globaux 
y_trainval_true_ols = []
y_trainval_pred_ols = []

#stocke les r² par split 
r2_in_split_ols = []
r2_oos_split_ols = []
y_true = []
y_pred_ols = []

#pour les portefeuilles (voir notebook results)
dates_ols = []
tickers_ols = []

#sucess ratio 
success_ratio_in_ols = []
success_ratio_oos_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    #R² in-sample
    y_trainval_pred = ols.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_ols.append(r2_in)

    #R² oos
    y_test_pred = ols.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_ols.append(r2_out)

    #Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_ols.append(sr_in)
    success_ratio_oos_ols.append(sr_out)

    #On stocke tout dans un tableau
    tickers_ols.append(x_test["Ticker"])
    dates_ols.append(x_test["Date"])
    y_true.append(y_test)
    y_pred_ols.append(y_test_pred)
    y_trainval_true_ols.append(y_trainval)
    y_trainval_pred_ols.append(y_trainval_pred) 

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")
    
#On concatène les résultats de tous les splits en un df
dates_ols = np.concatenate(dates_ols)
tickers_ols = np.concatenate(tickers_ols)
y_true = np.concatenate(y_true)
y_pred_ols = np.concatenate(y_pred_ols)

y_trainval_true_ols = np.concatenate(y_trainval_true_ols)
y_trainval_pred_ols = np.concatenate(y_trainval_pred_ols)

df_results_ols = pd.DataFrame({
    "Date": dates_ols,
    "Ticker": tickers_ols, 
    "y_true": y_true,
    "y_pred_ols": y_pred_ols,
})

r2_in_ols = r2(y_trainval_true_ols, y_trainval_pred_ols)
r2_oos_ols = r2(y_true, y_pred_ols)

print(f"\nR² in-sample OLS : {r2_in_ols}")
print(f"R² oos OLS : {r2_oos_ols}")

print(f"\nMoyenne Success ratio in-sample OLS : {np.mean(success_ratio_in_ols):.6f}")
print(f"Moyenne Success ratio oos OLS : {np.mean(success_ratio_oos_ols):.6f}")

100%|██████████| 3/3 [00:00<00:00, 38.75it/s]

R² in-sample : 0.038011 | R² oos : 0.014694
R² in-sample : 0.037695 | R² oos : 0.031914
R² in-sample : 0.037578 | R² oos : 0.023721

R² in-sample OLS : 0.037758268851299404
R² oos OLS : 0.02499260739484377

Moyenne Success ratio in-sample OLS : 0.572185
Moyenne Success ratio oos OLS : 0.581062


In [10]:
#HISTORICAL AVERAGE
r2_in_sample_ha = []
r2_oos_ha = []
success_ratio_in_ha = []
success_ratio_oos_ha = []

ha_pred = []
ha_true = []
dates_ha = []
tickers_ha = []

ha_trainval_true_all = []
ha_trainval_pred_all = []   

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits, start=1):

    tickers_trainval = pd.concat([x_train['Ticker'], x_val['Ticker']], ignore_index=True)
    y_trainval_all = pd.concat([y_train, y_val], ignore_index=True)
    trainval = pd.DataFrame({'Ticker': tickers_trainval, 'y': y_trainval_all})

    mean_by_ticker = trainval.groupby('Ticker')['y'].mean()

    preds_trainval = trainval['Ticker'].map(mean_by_ticker).values
    ha_trainval_true_all.extend(trainval['y'].values)
    ha_trainval_pred_all.extend(preds_trainval)

    r2_in = r2(trainval['y'].values, preds_trainval)
    sr_in = success_ratio(trainval['y'].values, preds_trainval)

    r2_in_sample_ha.append(r2_in)
    success_ratio_in_ha.append(sr_in)

    preds_split = [mean_by_ticker.get(tkr, np.nan) for tkr in x_test['Ticker']]
    preds_split = np.array(preds_split)
    r2_out = r2(y_test.values, preds_split)
    sr_out = success_ratio(y_test.values, preds_split)

    r2_oos_ha.append(r2_out)
    success_ratio_oos_ha.append(sr_out)

    ha_pred.extend(preds_split)
    ha_true.extend(y_test)
    dates_ha.append(x_test['Date'])
    tickers_ha.append(x_test['Ticker'])

    print(f"[Split {split_idx}] R² HA IN-sample: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

dates_ha = np.concatenate(dates_ha)
tickers_ha = np.concatenate(tickers_ha)
ha_pred = np.array(ha_pred)
ha_true = np.array(ha_true)

# === DataFrame final
df_results_ha = pd.DataFrame({
    "Date": dates_ha,
    "Ticker": tickers_ha,
    "y_true": ha_true,
    "y_pred_ha": ha_pred,
})

# Convertir en array
ha_trainval_true_all = np.array(ha_trainval_true_all)
ha_trainval_pred_all = np.array(ha_trainval_pred_all)

# Calculs globaux
r2_in_ha_global = r2(ha_trainval_true_all, ha_trainval_pred_all)
r2_oos_ha_global = r2(ha_true, ha_pred)


print("\nR² IN-sample global HA :", r2_in_ha_global)
print("R² OOS global HA :", r2_oos_ha_global)


[Split 1] R² HA IN-sample: 0.019580 | OOS: 0.021224 | SR IN: 0.566 | SR OOS: 0.560
[Split 2] R² HA IN-sample: 0.019650 | OOS: -0.000761 | SR IN: 0.566 | SR OOS: 0.559
[Split 3] R² HA IN-sample: 0.019109 | OOS: 0.018824 | SR IN: 0.565 | SR OOS: 0.597

R² IN-sample global HA : 0.019442077536028224
R² OOS global HA : 0.012188896170098107


ALGORITHMES

In [11]:
"""
PLS : Partial Least Squares
Hyperparamètres :
- k : nombre de composantes latentes, choisi pour minimiser la MSE sur la validation.
"""

# Pour calculer les R² globaux
y_trainval_true_pls = []
y_trainval_pred_pls = []

# Stocke les R² par split
r2_in_split_pls = []
r2_oos_split_pls = []
y_true_pls = []
y_pred_pls = []

# Pour les portefeuilles (voir notebook results)
dates_pls = []
tickers_pls = []

# Success ratio
success_ratio_in_pls = []
success_ratio_oos_pls = []

# Hyperparamètres PLS
best_components_list = []
mse_val_grids = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    # Recherche du meilleur k
    for k in candidate_ks:
        pls = PLSRegression(n_components=k, scale=False)
        pls.fit(x_train[covariates], y_train)
        y_val_pred = pls.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids.append(mse_val_grid)
    best_components_list.append(best_k)
    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    # Réentraîner sur train+val avec le meilleur k
    pls_final = PLSRegression(n_components=best_k, scale=False)
    pls_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = pls_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_pls.append(r2_in)

    # R² oos
    y_test_pred = pls_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_pls.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_pls.append(sr_in)
    success_ratio_oos_pls.append(sr_out)

    # Stockage pour global
    tickers_pls.append(x_test["Ticker"])
    dates_pls.append(x_test["Date"])
    y_true_pls.append(y_test)
    y_pred_pls.append(y_test_pred)
    y_trainval_true_pls.append(y_trainval)
    y_trainval_pred_pls.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_pls = np.concatenate(dates_pls)
tickers_pls = np.concatenate(tickers_pls)
y_true_pls = np.concatenate(y_true_pls)
y_pred_pls = np.concatenate(y_pred_pls)

y_trainval_true_pls = np.concatenate(y_trainval_true_pls)
y_trainval_pred_pls = np.concatenate(y_trainval_pred_pls)

df_results_pls = pd.DataFrame({
    "Date": dates_pls,
    "Ticker": tickers_pls,
    "y_true": y_true_pls,
    "y_pred_pls": y_pred_pls,
})

# R² globaux
r2_in_pls = r2(y_trainval_true_pls, y_trainval_pred_pls)
r2_oos_pls = r2(y_true_pls, y_pred_pls)

# Affichages
print(f"\nR² in-sample PLS : {r2_in_pls:.6f}")
print(f"R² oos PLS : {r2_oos_pls:.6f}")
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_pls):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_pls):.6f}")

print(f"\nMoyenne Success ratio in-sample PLS : {np.mean(success_ratio_in_pls):.6f}")
print(f"Moyenne Success ratio oos PLS : {np.mean(success_ratio_oos_pls):.6f}")

 33%|███▎      | 1/3 [00:04<00:09,  4.98s/it]

Split 1 : meilleur nombre de composantes k = 1
R² in-sample : 0.022679 | R² oos : 0.029549


 67%|██████▋   | 2/3 [00:09<00:04,  4.84s/it]

Split 2 : meilleur nombre de composantes k = 4
R² in-sample : 0.031472 | R² oos : 0.021320
Split 3 : meilleur nombre de composantes k = 12
R² in-sample : 0.037575 | R² oos : 0.023761


100%|██████████| 3/3 [00:15<00:00,  5.01s/it]


R² in-sample PLS : 0.030687
R² oos PLS : 0.023963
Moyenne des R² in-sample (splits) : 0.030575
Moyenne des R² oos (splits) : 0.024877

Moyenne Success ratio in-sample PLS : 0.568567
Moyenne Success ratio oos PLS : 0.574566


In [ ]:
"""
PCR : Principal Component Regression
Hyperparamètre : k (nombre de composantes principales)
"""

# Pour calculer les R² globaux
y_trainval_true_pcr = []
y_trainval_pred_pcr = []

# Stocke les R² par split
r2_in_split_pcr = []
r2_oos_split_pcr = []
y_true_pcr = []
y_pred_pcr = []

# Pour les portefeuilles (voir notebook results)
dates_pcr = []
tickers_pcr = []

# Success ratio
success_ratio_in_pcr = []
success_ratio_oos_pcr = []

# Hyperparamètres spécifiques PCR
best_components_pcr = []
mse_val_grids_pcr = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    # Recherche du meilleur k
    for k in candidate_ks:
        pcr_pipe = Pipeline([
            ('pca', PCA(n_components=k)),
            ('reg', LinearRegression())
        ])
        pcr_pipe.fit(x_train[covariates], y_train)
        y_val_pred = pcr_pipe.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids_pcr.append(mse_val_grid)
    best_components_pcr.append(best_k)
    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    # Réentraîner sur train+val avec le meilleur k
    pcr_final = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = pcr_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_pcr.append(r2_in)

    # R² oos
    y_test_pred = pcr_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_pcr.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_pcr.append(sr_in)
    success_ratio_oos_pcr.append(sr_out)

    # Stockage pour global
    tickers_pcr.append(x_test["Ticker"])
    dates_pcr.append(x_test["Date"])
    y_true_pcr.append(y_test)
    y_pred_pcr.append(y_test_pred)
    y_trainval_true_pcr.append(y_trainval)
    y_trainval_pred_pcr.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
tickers_pcr = np.concatenate(tickers_pcr)
dates_pcr = np.concatenate(dates_pcr)
y_true_pcr = np.concatenate(y_true_pcr)
y_pred_pcr = np.concatenate(y_pred_pcr)
y_trainval_true_pcr = np.concatenate(y_trainval_true_pcr)
y_trainval_pred_pcr = np.concatenate(y_trainval_pred_pcr)

df_results_pcr = pd.DataFrame({
    "Date": dates_pcr,
    "Ticker": tickers_pcr,
    "y_true": y_true_pcr,
    "y_pred_pcr": y_pred_pcr,
})

# R² globaux
r2_in_pcr = r2(y_trainval_true_pcr, y_trainval_pred_pcr)
r2_oos_pcr = r2(y_true_pcr, y_pred_pcr)

# Affichages
print(f"\nR² in-sample PCR : {r2_in_pcr:.6f}")
print(f"R² oos PCR : {r2_oos_pcr:.6f}")
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_pcr):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_pcr):.6f}")

print(f"\nMoyenne Success ratio in-sample PCR : {np.mean(success_ratio_in_pcr):.6f}")
print(f"Moyenne Success ratio oos PCR : {np.mean(success_ratio_oos_pcr):.6f}")


  0%|          | 0/3 [00:00<?, ?it/s]


NameError: name 'Pipeline' is not defined

In [13]:
"""
ENet : Elastic Net

Hyperparamètres :
- lambda (alpha) : coefficient de pénalisation choisi pour minimiser la MSE
- l1_ratio fixé à 0.5
"""

# Pour calculer les R² globaux
y_trainval_true_en = []
y_trainval_pred_en = []

# Stocke les R² par split
r2_in_split_en = []
r2_oos_split_en = []
y_true_en = []
y_pred_en = []

# Pour les portefeuilles
dates_en = []
tickers_en = []

# Success ratio
success_ratio_in_en = []
success_ratio_oos_en = []

# Hyperparamètres spécifiques
best_lambdas = []

enet_param_grid = {
    'alpha': np.logspace(np.log10(0.0008), np.log10(0.0005), num=12)
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_lambda = None

    # Recherche du meilleur alpha
    for params in ParameterGrid(enet_param_grid):
        enet = ElasticNet(**params, l1_ratio=0.5, max_iter=10000)
        enet.fit(x_train[covariates], y_train)
        y_val_pred = enet.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        if mse < best_mse:
            best_mse = mse
            best_lambda = params['alpha']

    best_lambdas.append(best_lambda)
    print(f"Split {split_idx} : meilleur lambda = {best_lambda}")

    # Réentraîner sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    en_final = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = en_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_en.append(r2_in)

    # R² oos
    y_test_pred = en_final.predict(x_test[covariates])
    r2_out = r2(y_test.values, y_test_pred)
    r2_oos_split_en.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_en.append(sr_in)
    success_ratio_oos_en.append(sr_out)

    # Stockage pour global
    tickers_en.append(x_test["Ticker"])
    dates_en.append(x_test["Date"])
    y_true_en.append(y_test)
    y_pred_en.append(y_test_pred)
    y_trainval_true_en.append(y_trainval)
    y_trainval_pred_en.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_en = np.concatenate(dates_en)
tickers_en = np.concatenate(tickers_en)
y_true_en = np.concatenate(y_true_en)
y_pred_en = np.concatenate(y_pred_en)
y_trainval_true_en = np.concatenate(y_trainval_true_en)
y_trainval_pred_en = np.concatenate(y_trainval_pred_en)

df_results_en = pd.DataFrame({
    "Date": dates_en,
    "Ticker": tickers_en,
    "y_true": y_true_en,
    "y_pred_en": y_pred_en,
})

# R² globaux
r2_in_en = r2(y_trainval_true_en, y_trainval_pred_en)
r2_oos_en = r2(y_true_en, y_pred_en)

# Affichages
print(f"\nR² in-sample ENet : {r2_in_en:.6f}")
print(f"R² oos ENet : {r2_oos_en:.6f}")
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_en):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_en):.6f}")

print(f"\nMoyenne Success ratio in-sample ENet : {np.mean(success_ratio_in_en):.6f}")
print(f"Moyenne Success ratio oos ENet : {np.mean(success_ratio_oos_en):.6f}")


  0%|          | 0/3 [00:00<?, ?it/s]

Split 1 : meilleur lambda = 0.0008000000000000004


 33%|███▎      | 1/3 [00:25<00:51, 25.58s/it]

R² in-sample : 0.032710 | R² oos : 0.038142
Split 2 : meilleur lambda = 0.0008000000000000004


 67%|██████▋   | 2/3 [01:09<00:36, 36.13s/it]

R² in-sample : 0.032414 | R² oos : 0.026124
Split 3 : meilleur lambda = 0.0004999999999999999


100%|██████████| 3/3 [01:56<00:00, 38.99s/it]

R² in-sample : 0.035389 | R² oos : 0.023557

R² in-sample ENet : 0.033528
R² oos ENet : 0.027220
Moyenne des R² in-sample (splits) : 0.033505
Moyenne des R² oos (splits) : 0.029274

Moyenne Success ratio in-sample ENet : 0.572997
Moyenne Success ratio oos ENet : 0.579690


In [14]:
"""
RF : Random Forest
Hyperparamètres :
- n_estimators : nombre d’arbres dans la forêt.
- max_depth : profondeur maximale de chaque arbre.
- min_samples_leaf : nombre minimal d’échantillons dans une feuille.
- max_features : nombre de variables considérées pour le split.
"""

param_grid_rf = {
    'n_estimators': [100, 125, 150],
    'max_depth': [5, 6, 7],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['log2', None]
}

# Pour calculer les R² globaux
y_trainval_true_rf = []
y_trainval_pred_rf = []

# Stocke les R² par split
r2_in_split_rf = []
r2_oos_split_rf = []
y_true_rf = []
y_pred_rf = []

# Pour les portefeuilles
dates_rf = []
tickers_rf = []

# Success ratio
success_ratio_in_rf = []
success_ratio_oos_rf = []

# Hyperparamètres spécifiques
best_params_rf = []
mse_val_grids_rf = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_rf):
        rf = RandomForestRegressor(
            **params,
            n_jobs=-1,
            random_state=0
        )
        rf.fit(x_train[covariates], y_train)
        y_val_pred = rf.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_rf.append(mse_grid)
    best_params_rf.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    rf_final = RandomForestRegressor(
        **best_params,
        n_jobs=-1,
        random_state=0
    )
    rf_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = rf_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_rf.append(r2_in)

    # R² oos
    y_test_pred = rf_final.predict(x_test[covariates])
    r2_out = r2(y_test.values, y_test_pred)
    r2_oos_split_rf.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_rf.append(sr_in)
    success_ratio_oos_rf.append(sr_out)

    # Stockage pour global
    tickers_rf.append(x_test["Ticker"])
    dates_rf.append(x_test["Date"])
    y_true_rf.append(y_test)
    y_pred_rf.append(y_test_pred)
    y_trainval_true_rf.append(y_trainval)
    y_trainval_pred_rf.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_rf = np.concatenate(dates_rf)
tickers_rf = np.concatenate(tickers_rf)
y_true_rf = np.concatenate(y_true_rf)
y_pred_rf = np.concatenate(y_pred_rf)
y_trainval_true_rf = np.concatenate(y_trainval_true_rf)
y_trainval_pred_rf = np.concatenate(y_trainval_pred_rf)

df_results_rf = pd.DataFrame({
    "Date": dates_rf,
    "Ticker": tickers_rf,
    "y_true": y_true_rf,
    "y_pred_rf": y_pred_rf,
})

# R² globaux
r2_in_rf = r2(y_trainval_true_rf, y_trainval_pred_rf)
r2_oos_rf = r2(y_true_rf, y_pred_rf)

# Affichages
print(f"\nR² in-sample RF : {r2_in_rf:.6f}")
print(f"R² oos RF : {r2_oos_rf:.6f}")
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_rf):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_rf):.6f}")

print(f"\nMoyenne Success ratio in-sample RF : {np.mean(success_ratio_in_rf):.6f}")
print(f"Moyenne Success ratio oos RF : {np.mean(success_ratio_oos_rf):.6f}")


  0%|          | 0/3 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'max_depth': 7, 'max_features': 'log2', 'min_samples_leaf': 3, 'n_estimators': 150} (MSE val = 0.002981)


 33%|███▎      | 1/3 [02:53<05:46, 173.27s/it]

R² in-sample : 0.122605 | R² oos : 0.048871

Split 2 : meilleurs params = {'max_depth': 7, 'max_features': None, 'min_samples_leaf': 1, 'n_estimators': 100} (MSE val = 0.003415)


 67%|██████▋   | 2/3 [05:37<02:48, 168.11s/it]

R² in-sample : 0.167922 | R² oos : 0.034596

Split 3 : meilleurs params = {'max_depth': 7, 'max_features': None, 'min_samples_leaf': 1, 'n_estimators': 100} (MSE val = 0.006784)


100%|██████████| 3/3 [08:21<00:00, 167.23s/it]

R² in-sample : 0.166011 | R² oos : 0.025600

R² in-sample RF : 0.152473
R² oos RF : 0.033217
Moyenne des R² in-sample (splits) : 0.152180
Moyenne des R² oos (splits) : 0.036356

Moyenne Success ratio in-sample RF : 0.595437
Moyenne Success ratio oos RF : 0.579702


In [15]:
"""
GBRT : Gradient Boosted Regression Trees
Hyperparamètres :
- n_estimators : nombre d’arbres successifs
- learning_rate : taux d’apprentissage
- max_depth : profondeur maximale des arbres
- loss : fonction de perte
- alpha : paramètre huber
"""

param_grid_gbrt = {
    'n_estimators': [200],
    'learning_rate': [0.01],
    'max_depth': [2, 3, 4],
    'loss': ['huber'],
    'alpha': [0.9]
}

# Pour calculer les R² globaux
y_trainval_true_gbrt = []
y_trainval_pred_gbrt = []

# Stocke les R² par split
r2_in_split_gbrt = []
r2_oos_split_gbrt = []
y_true_gbrt = []
y_pred_gbrt = []

# Pour les portefeuilles
dates_gbrt = []
tickers_gbrt = []

# Success ratio
success_ratio_in_gbrt = []
success_ratio_oos_gbrt = []

# Hyperparamètres spécifiques
best_params_gbrt = []
mse_val_grids_gbrt = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_gbrt):
        gbrt = GradientBoostingRegressor(**params, random_state=0)
        gbrt.fit(x_train[covariates], y_train)
        y_val_pred = gbrt.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_gbrt.append(mse_grid)
    best_params_gbrt.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    gbrt_final = GradientBoostingRegressor(**best_params, random_state=0)
    gbrt_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = gbrt_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_gbrt.append(r2_in)

    # R² oos
    y_test_pred = gbrt_final.predict(x_test[covariates])
    r2_out = r2(y_test.values, y_test_pred)
    r2_oos_split_gbrt.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_gbrt.append(sr_in)
    success_ratio_oos_gbrt.append(sr_out)

    # Stockage pour global
    tickers_gbrt.append(x_test["Ticker"])
    dates_gbrt.append(x_test["Date"])
    y_true_gbrt.append(y_test)
    y_pred_gbrt.append(y_test_pred)
    y_trainval_true_gbrt.append(y_trainval)
    y_trainval_pred_gbrt.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_gbrt = np.concatenate(dates_gbrt)
tickers_gbrt = np.concatenate(tickers_gbrt)
y_true_gbrt = np.concatenate(y_true_gbrt)
y_pred_gbrt = np.concatenate(y_pred_gbrt)
y_trainval_true_gbrt = np.concatenate(y_trainval_true_gbrt)
y_trainval_pred_gbrt = np.concatenate(y_trainval_pred_gbrt)

df_results_gbrt = pd.DataFrame({
    "Date": dates_gbrt,
    "Ticker": tickers_gbrt,
    "y_true": y_true_gbrt,
    "y_pred_gbrt": y_pred_gbrt,
})

# R² globaux
r2_in_gbrt = r2(y_trainval_true_gbrt, y_trainval_pred_gbrt)
r2_oos_gbrt = r2(y_true_gbrt, y_pred_gbrt)

# Affichages
print(f"\nR² in-sample GBRT : {r2_in_gbrt:.6f}")
print(f"R² oos GBRT : {r2_oos_gbrt:.6f}")
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_gbrt):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_gbrt):.6f}")

print(f"\nMoyenne Success ratio in-sample GBRT : {np.mean(success_ratio_in_gbrt):.6f}")
print(f"Moyenne Success ratio oos GBRT : {np.mean(success_ratio_oos_gbrt):.6f}")


  0%|          | 0/3 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 2, 'n_estimators': 200} (MSE val = 0.003117)


 33%|███▎      | 1/3 [01:17<02:34, 77.45s/it]

R² in-sample : 0.028418 | R² oos : 0.035103


 33%|███▎      | 1/3 [02:14<04:29, 134.66s/it]


KeyboardInterrupt: 

In [ ]:
"""
XGB : XGBoost Regressor
Hyperparamètres :
- n_estimators : nombre d’arbres
- max_depth : profondeur max
- eta : learning rate
"""

param_grid_xgb = {
    'n_estimators': [150, 200, 250],
    'max_depth': [3, 4],
    'eta': [0.01, 0.02],
}

# Pour calculer les R² globaux
y_trainval_true_xgb = []
y_trainval_pred_xgb = []

# Stocke les R² par split
r2_in_split_xgb = []
r2_oos_split_xgb = []
y_true_xgb = []
y_pred_xgb = []

# Pour les portefeuilles
dates_xgb = []
tickers_xgb = []

# Success ratio
success_ratio_in_xgb = []
success_ratio_oos_xgb = []

# Hyperparamètres spécifiques
best_params_xgb = []
mse_val_grids_xgb = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_xgb):
        xgb_model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        xgb_model.fit(x_train[covariates], y_train)
        y_val_pred = xgb_model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_xgb.append(mse_grid)
    best_params_xgb.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    xgb_final = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    xgb_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = xgb_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_xgb.append(r2_in)

    # R² oos
    y_test_pred = xgb_final.predict(x_test[covariates])
    r2_out = r2(y_test.values, y_test_pred)
    r2_oos_split_xgb.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_xgb.append(sr_in)
    success_ratio_oos_xgb.append(sr_out)

    # Stockage pour global
    tickers_xgb.append(x_test["Ticker"])
    dates_xgb.append(x_test["Date"])
    y_true_xgb.append(y_test)
    y_pred_xgb.append(y_test_pred)

    y_trainval_true_xgb.append(y_trainval)
    y_trainval_pred_xgb.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_xgb = np.concatenate(dates_xgb)
tickers_xgb = np.concatenate(tickers_xgb)
y_true_xgb = np.concatenate(y_true_xgb)
y_pred_xgb = np.concatenate(y_pred_xgb)
y_trainval_true_xgb = np.concatenate(y_trainval_true_xgb)
y_trainval_pred_xgb = np.concatenate(y_trainval_pred_xgb)

df_results_xgb = pd.DataFrame({
    "Date": dates_xgb,
    "Ticker": tickers_xgb,
    "y_true": y_true_xgb,
    "y_pred_xgb": y_pred_xgb,
})

# R² globaux
r2_in_xgb = r2(y_trainval_true_xgb, y_trainval_pred_xgb)
r2_oos_xgb = r2(y_true_xgb, y_pred_xgb)

# Affichages
print(f"\nR² in-sample XGB : {r2_in_xgb:.6f}")
print(f"R² oos XGB : {r2_oos_xgb:.6f}")
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_xgb):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_xgb):.6f}")

print(f"\nMoyenne Success ratio in-sample XGB : {np.mean(success_ratio_in_xgb):.6f}")
print(f"Moyenne Success ratio oos XGB : {np.mean(success_ratio_oos_xgb):.6f}")


  0%|          | 0/3 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'eta': 0.01, 'max_depth': 3, 'n_estimators': 200} (MSE val = 0.003031)


 33%|███▎      | 1/3 [00:08<00:17,  8.76s/it]

R² in-sample : 0.051238 | R² oos : 0.048011

Split 2 : meilleurs params = {'eta': 0.02, 'max_depth': 3, 'n_estimators': 150} (MSE val = 0.003424)


 67%|██████▋   | 2/3 [00:18<00:09,  9.42s/it]

R² in-sample : 0.062373 | R² oos : 0.036395

Split 3 : meilleurs params = {'eta': 0.02, 'max_depth': 3, 'n_estimators': 250} (MSE val = 0.006712)


100%|██████████| 3/3 [00:29<00:00,  9.93s/it]

R² in-sample : 0.080390 | R² oos : 0.019785

R² in-sample XGB : 0.064895
R² oos XGB : 0.031085
Moyenne des R² in-sample (splits) : 0.064667
Moyenne des R² oos (splits) : 0.034730

Moyenne Success ratio in-sample XGB : 0.579858
Moyenne Success ratio oos XGB : 0.576411


METRIQUES

In [ ]:
import pandas as pd

df_global_metrics = pd.DataFrame([
    {"Model": "HA",   "R2_in": r2_in_ha_global,   "R2_oos": r2_oos_ha_global},
    {"Model": "OLS",  "R2_in": r2_in_ols,         "R2_oos": r2_oos_ols},
    {"Model": "PLS",  "R2_in": r2_in_pls,         "R2_oos": r2_oos_pls},
    {"Model": "PCR",  "R2_in": r2_in_pcr,         "R2_oos": r2_oos_pcr},
    {"Model": "ENet", "R2_in": r2_in_en,          "R2_oos": r2_oos_en},
    {"Model": "RF",   "R2_in": r2_in_rf,          "R2_oos": r2_oos_rf},
    {"Model": "GBRT", "R2_in": r2_in_gbrt,        "R2_oos": r2_oos_gbrt},
    {"Model": "XGB",  "R2_in": r2_in_xgb,         "R2_oos": r2_oos_xgb},
])

print("R² globaux")
print(df_global_metrics)

print("\nR² IN-sample global HA :", r2_in_ha_global)
print("R² OOS global HA :", r2_oos_ha_global)

R² globaux
  Model     R2_in    R2_oos
0    HA  0.019442  0.012189
1   OLS  0.037758  0.024993
2   PLS  0.030687  0.023963
3   PCR  0.028940  0.022647
4  ENet  0.033528  0.027220
5    RF  0.152473  0.033217
6  GBRT  0.051498  0.023290
7   XGB  0.064895  0.031085

R² IN-sample global HA : 0.019442077536028224
R² OOS global HA : 0.012188896170098107


PORTEFEUILLES

In [ ]:
#prédictions : all model 
df_predict = df_results_ols[["Date", "Ticker"]].copy()
df_predict = df_predict.merge(df_results_pls, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_pcr, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_en, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_rf, on=["Date", "Ticker"], how="inner")

III. RESULTATS

Moyenne des R² in-sample PLS : 0.030965
Moyenne des R² oos PLS : 0.025964
Moyenne des R² in-sample (OLS) : 0.038000
Moyenne des R² oos (OLS) : 0.024593
Moyenne des R² in-sample (PCR) : 0.029210
Moyenne des R² oos (PCR) : 0.024018
Moyenne des R² in-sample (ENet) : 0.033881
Moyenne des R² oos (ENet) : 0.029896
Moyenne des R² in-sample (RF) : 0.135247
Moyenne des R² oos (RF) : 0.028310
Moyenne des R² in-sample (GBRT) : 0.040733
Moyenne des R² oos (GBRT) : 0.025664


In [ ]:
#PREMIER TABLEAU : R² OOS Monthly

#* 100 pcq on est en décimal 
R2_monthly_sum = {
    'OLS' : df_r2_monthly["R2_OLS"].mean() * 100,
    'PLS' : df_r2_monthly_pls["R2_PLS"].mean() * 100,
    'PCR' : df_r2_monthly_pcr["R2_PCR"].mean() * 100, 
    'Enet' : df_r2_monthly_en["R2_ENET"].mean() * 100,
    'RF' : df_r2_monthly_rf["R2_RF"].mean() * 100,
    'GBRT' : df_r2_monthly_gbrt["R2_GBRT"].mean() * 100
}


#tableau 
# Transformer le dictionnaire en DataFrame pour un tableau
df_table = pd.DataFrame.from_dict(R2_monthly_sum, orient='index', columns=['Mean_R2_OOS'])
print("=== Tableau résumé R² OOS Monthly ===")
print(df_table)



NameError: name 'df_r2_monthly' is not defined

In [ ]:
import matplotlib.pyplot as plt

models = list(R2_monthly_sum.keys())
values = list(R2_monthly_sum.values())

plt.figure(figsize=(8,5))
bars = plt.bar(models, values, edgecolor='black', width=0.35)  # width < 1 pour des barres plus fines

# Titre et labels
plt.title('Comparaison des modèles (moyenne mensuelle)', fontsize=14, fontweight='bold')
plt.ylabel('Mean monthly $R^2_{OOS}$ (%)', fontsize=12)

# Valeurs au-dessus des barres
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.05,
             f'{height:.2f}', ha='center', va='bottom', fontsize=10)

# Couleur personnalisée
for bar in bars:
    bar.set_color('#4C72B0')

# Style épuré
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
#R2 benchmark HA 

def compute_oos_r_square(actual, y_benchmark, y_pred):
    MSFE_benchmark = mean_squared_error(y_benchmark, actual)
    MSFE_pred = mean_squared_error(y_pred, actual)
    return 1 - MSFE_pred / MSFE_benchmark

In [ ]:
#HA Benchmark 
actual_test = all_y_true_ols          # identique pour tous les modèles
y_pred_HA = all_y_pred_HA

#Prédictions concaténées dans un dictionnaire
model_predictions = {
    "OLS": all_y_pred_ols,
    "PLS": all_y_pred_pls,
    "PCR": all_y_pred_pcr,
    "ENet": all_y_pred_en,
    "RF": all_y_pred_rf,
    "GBRT": all_y_pred_gbrt
}

#Calcul des métriques
results = []

for model_name, y_pred in model_predictions.items():
    r2_oos = compute_oos_r_square(actual_test, y_pred_HA, y_pred) * 100  # en %
    results.append([model_name, r2_oos])

results.append(["HA", 0.0])

#df final 
df_r2_oos = pd.DataFrame(results, columns=["Model", "OOS_R2(%)"])

print(df_r2_oos)